# Trace `optimize_anything` API tutorial

[Open in Colab](https://colab.research.google.com/github/doxav/NewTrace/blob/experimental/examples/notebooks/optimize_anything_api.ipynb)

This notebook demonstrates the additive `opto.optimize_anything` compatibility layer and compares it with native Trace. It starts with deterministic offline examples, then runs low-budget GPT-5 nano examples when OpenAI/OpenRouter credentials are configured. The examples are GEPA-style, but are tutorial examples inspired by public optimize-anything workflows rather than claims about any exact current blog implementation.


In [29]:
import os, sys, json, textwrap
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    # Install the branch version because the new opto.optimize_anything API
    # may not exist yet in the published trace-opt package.
    trace_ref = os.getenv("TRACE_NOTEBOOK_REF", "experimental")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"git+https://github.com/doxav/NewTrace.git@{trace_ref}",
        "datasets", "litellm",
    ])
print("Python", sys.version.split()[0], "Colab", IN_COLAB)


Python 3.12.13 Colab False


## Configure OpenRouter or OpenAI

The cell uses Colab secrets if available (`OPENROUTER_API_KEY`, `OPENAI_API_KEY`), then normal environment variables. OpenRouter uses LiteLLM's `openrouter/...` model convention and OpenAI defaults to `gpt-5-nano`.


In [30]:
def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

openrouter_key = _colab_secret("OPENROUTER_API_KEY") or os.getenv("OPENROUTER_API_KEY")
openai_key = _colab_secret("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
os.environ.setdefault("TRACE_DEFAULT_LLM_BACKEND", "LiteLLM")

if openrouter_key:
    os.environ["OPENROUTER_API_KEY"] = openrouter_key
    os.environ["OPENAI_API_KEY"] = openrouter_key
    os.environ.setdefault("OPENAI_API_BASE", "https://openrouter.ai/api/v1")
    os.environ.setdefault("TRACE_LITELLM_MODEL", os.getenv("OPENROUTER_MODEL", "openrouter/openai/gpt-4o-mini"))
    provider = "OpenRouter"
elif openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    os.environ["TRACE_LITELLM_MODEL"] = "gpt-4o-mini"
    provider = "OpenAI"
else:
    provider = "offline"

HAS_LLM = provider != "offline"
print({"provider": provider, "model": os.getenv("TRACE_LITELLM_MODEL")})


{'provider': 'OpenAI', 'model': 'gpt-4o-mini'}


In [31]:
import opto.optimize_anything as oa
from opto.optimize_anything import TraceOptimizerBackend
from opto.trace import node, bundle, GRAPH
from opto.optimizers import OptoPrimeV2, OptoPrime, OPROv2, TextGrad
try:
    from opto.optimizers import OptoPrimeMulti
except Exception:
    OptoPrimeMulti = None
print("optimize_anything ready", oa.EngineConfig(max_metric_calls=3))

optimize_anything ready EngineConfig(max_metric_calls=3, max_steps=None, higher_is_better=True, cache_evaluation=True, capture_stdio=False, candidate_selection_strategy='best', frontier_type='score', random_seed=0)


## Deterministic GEPA-style prompt optimization

A candidate can be a string, dict, or JSON-like object. Evaluators can call `oa.log()`; logs are captured in evaluation records rather than printed during evaluation.


In [32]:
train_examples = [
    {"question": "2 + 2", "answer": "4"},
    {"question": "3 * 3", "answer": "9"},
    {"question": "10 - 7", "answer": "3"},
]

def deterministic_prompt_evaluator(candidate, example, opt_state=None):
    prompt = candidate if isinstance(candidate, str) else candidate.get("prompt", "")
    score = 0.2
    if "calculate" in prompt.lower() or "solve" in prompt.lower(): score += 0.4
    if "answer only" in prompt.lower() or "final answer" in prompt.lower(): score += 0.4
    oa.log("prompt_len", len(prompt), "question", example["question"])
    return min(score, 1.0), {"scores": {"prompt_proxy": score}}

def deterministic_proposer(candidate, feedback, **kwargs):
    if "answer only" not in candidate.lower():
        return candidate + "\nCalculate carefully and answer only with the final answer."
    return candidate

result = oa.optimize_anything(
    seed_candidate="You are a helpful assistant.",
    evaluator=deterministic_prompt_evaluator,
    dataset=train_examples,
    objective="Improve exact-answer arithmetic prompt quality.",
    config=oa.GEPAConfig(
        engine=oa.EngineConfig(max_metric_calls=12, max_steps=2, capture_stdio=True),
        reflection=oa.ReflectionConfig(custom_candidate_proposer=deterministic_proposer),
    ),
)
print("best_score", result.best_score)
print(result.best_candidate)
print("first_record_logs", result.history[0].logs)


best_score 1.0
You are a helpful assistant.
Calculate carefully and answer only with the final answer.
first_record_logs ['prompt_len 28 question 2 + 2']


## Trace optimizer backend

`TraceOptimizerBackend` adapts Trace optimizers (`OptoPrimeV2`, `OptoPrime`, `OptoPrimeMulti`, `OPROv2`, `TextGrad`, or custom protocol-compatible classes) to the proposer interface. The live cell is skipped without credentials.


In [33]:
if HAS_LLM:
    trace_backend = TraceOptimizerBackend(
        optimizer_cls="OptoPrimeV2",
        optimizer_kwargs={"memory_size": 1, "use_json_object_format": False},
    )
    llm_result = oa.optimize_anything(
        seed_candidate="You are a helpful assistant.",
        evaluator=deterministic_prompt_evaluator,
        dataset=train_examples[:1],
        objective="Make the prompt concise and exact-answer oriented.",
        config=oa.GEPAConfig(
            engine=oa.EngineConfig(max_metric_calls=2, max_steps=1, capture_stdio=True),
            reflection=oa.ReflectionConfig(custom_candidate_proposer=trace_backend),
        ),
    )
    print(type(llm_result.best_candidate), llm_result.best_score)
    print(str(llm_result.best_candidate)[:300])
else:
    print("Skipping live TraceOptimizerBackend demo: no OPENAI_API_KEY/OPENROUTER_API_KEY configured.")


<class 'str'> 0.2
You are a helpful assistant.


## Native Trace API comparison

The native API is graph-first: `node -> bundle -> optimizer.backward -> optimizer.step`. The compatibility API is evaluator-first: `candidate -> evaluator -> proposer`.


In [34]:
@bundle()
def evaluate_prompt_text(prompt, question):
    return prompt + "\nQuestion: " + question

GRAPH.clear()
prompt = node("You are a helpful assistant.", trainable=True, description="Arithmetic answer prompt")
output = evaluate_prompt_text(prompt, "2 + 2")
optimizer = OptoPrimeV2([prompt], use_json_object_format=False, memory_size=1, max_tokens=256)
optimizer.zero_feedback()
optimizer.backward(output, "The answer should be concise and answer only with the final number.")

if HAS_LLM:
    update = optimizer.step(bypassing=True)
    print({k.name: str(v)[:200] for k, v in update.items()})
else:
    summary = optimizer.summarize()
    system_prompt, user_prompt = optimizer.construct_prompt(summary)
    print(system_prompt.splitlines()[0])
    print(user_prompt[:300])


{'str:0': '4'}


## BBEH / BBH-style task selection

Set `BBEH_TASK`, and optionally `BBEH_HF_DATASET`/`BBEH_SPLIT`, to load a HuggingFace dataset. Local mini tasks keep the notebook runnable offline.


In [35]:
FALLBACK_TASKS = {
    "boolean_expressions": [
        {"input": "not ( true and false )", "target": "true"},
        {"input": "true and false", "target": "false"},
    ],
    "date_understanding": [
        {"input": "Today is Monday. What day is tomorrow?", "target": "Tuesday"},
        {"input": "Yesterday was Friday. What day is today?", "target": "Saturday"},
    ],
    "word_sorting": [
        {"input": "Sort: zebra apple lemon", "target": "apple lemon zebra"},
        {"input": "Sort: beta alpha gamma", "target": "alpha beta gamma"},
    ],
}

def load_bbeh_like_task(task_name="boolean_expressions", n=8):
    dataset_id = os.getenv("BBEH_HF_DATASET")
    if dataset_id:
        try:
            from datasets import load_dataset
            return list(load_dataset(dataset_id, task_name, split=os.getenv("BBEH_SPLIT", "test")))[:n]
        except Exception as exc:
            print("Falling back to local examples:", exc)
    return FALLBACK_TASKS.get(task_name, FALLBACK_TASKS["boolean_expressions"])[:n]

task_name = os.getenv("BBEH_TASK", "boolean_expressions")
task_examples = load_bbeh_like_task(task_name)
print(task_name, task_examples)


boolean_expressions [{'input': 'not ( true and false )', 'target': 'true'}, {'input': 'true and false', 'target': 'false'}]


In [36]:
def bbeh_style_evaluator(candidate, example):
    prompt = candidate if isinstance(candidate, str) else candidate.get("prompt", "")
    lower = prompt.lower()
    score = 0.1
    if "think" in lower or "reason" in lower: score += 0.35
    if "answer only" in lower or "final answer" in lower: score += 0.45
    if task_name.replace("_", " ") in lower: score += 0.10
    return min(score, 1.0), {"scores": {"prompt_proxy": score}, "task": task_name}

def bbeh_tutorial_proposer(candidate, feedback, **kwargs):
    return (candidate + f"\nFor {task_name}, reason briefly, then provide the final answer only.").strip()

bbeh_result = oa.optimize_anything(
    seed_candidate="Solve the task.",
    evaluator=bbeh_style_evaluator,
    dataset=task_examples,
    objective=f"Improve performance on {task_name}.",
    config=oa.GEPAConfig(
        engine=oa.EngineConfig(max_metric_calls=8, max_steps=2),
        reflection=oa.ReflectionConfig(custom_candidate_proposer=bbeh_tutorial_proposer),
    ),
)
print(bbeh_result.best_score)
print(bbeh_result.best_candidate)


0.8999999999999999
Solve the task.
For boolean_expressions, reason briefly, then provide the final answer only.


## Optional `OptoPrimeMulti`

`OptoPrimeMulti` is available as a multi-candidate backend, but it is not the default backend. The cell uses tiny generation settings and skips without credentials.


In [ ]:
if HAS_LLM and OptoPrimeMulti is not None:
    multi_backend = TraceOptimizerBackend(
        optimizer_cls="OptoPrimeMulti",
        optimizer_kwargs={"num_responses": 2, "max_tokens": 256},
    )
    multi_result = oa.optimize_anything(
        seed_candidate="Solve the task.",
        evaluator=bbeh_style_evaluator,
        dataset=task_examples[:1],
        objective=f"Improve performance on {task_name}.",
        config=oa.GEPAConfig(
            engine=oa.EngineConfig(max_metric_calls=2, max_steps=1),
            reflection=oa.ReflectionConfig(custom_candidate_proposer=multi_backend),
        ),
    )
    print(f"type: {type(multi_result.best_candidate)}, score: {multi_result.best_score}")
else:
    print("Skipping OptoPrimeMulti backend demo.")


<class 'str'> 0.2
